# Metric Definitions & Sanity Checks

This notebook documents the analytical logic behind the Shiny dashboard and verifies it on the real data. It **reuses the same code** that the app uses (`src/`), so the numbers here match the dashboard exactly.

Run from the repo root.

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '..')  # so `src` is importable when running from Notebooks/
import pandas as pd, numpy as np
from src import data as d, metrics, geo
df = d.load_data()
print(df.shape, '|', df.year.min(), '-', df.year.max())
df.head(3)

## 1. Topic buckets
Each paper is mapped to one canonical AI topic by keyword matching on its subfield/topic text (first matching rule wins; see `src/data.py::TOPIC_RULES`).

In [ ]:
df.topic_bucket.value_counts()

## 2. Growth metrics (Growth tab)
YoY growth, YoY acceleration (pp) and CAGR over the period.

In [ ]:
counts = df.groupby('year').size().sort_index()
summary = pd.DataFrame({
    'papers': counts,
    'yoy_%': metrics.yoy_growth(counts).round(1),
    'yoy_accel_pp': metrics.yoy_acceleration(counts).round(1),
})
print('CAGR over period: {:.1f}%'.format(metrics.cagr(counts.iloc[0], counts.iloc[-1], len(counts)-1)*100))
summary

## 3. Impact metrics (Impact tab)
Median citations, low-citation share, top-1% citation share and the structural novelty proxy.

In [ ]:
med = df.citation_count.median()
c = np.sort(df.citation_count.values)[::-1]; n = max(1, int(np.ceil(len(c)*0.01)))
print('Median citations      :', med)
print('Low-citation share    : {:.0%} (below median)'.format((df.citation_count < med).mean()))
print('Top 1% citation share : {:.1f}%'.format(c[:n].sum()/c.sum()*100))
print('Mean novelty proxy    : {:.2f}'.format(df.novelty_proxy.mean()))

## 4. Concentration metrics (Concentration tab)
Top-5 country share, topic entropy (bits) and citation Gini, plus the per-country table.

In [ ]:
ex = d.explode_countries(df)
country_papers = ex.groupby('country').size().sort_values(ascending=False)
print('Top-5 country share : {:.0f}%'.format(metrics.top_n_share(country_papers.values, 5)))
print('Topic entropy       : {:.2f} bits'.format(metrics.shannon_entropy(df.topic_bucket.value_counts().values)))
print('Citation Gini       : {:.2f}'.format(metrics.gini(df.citation_count.values)))
country_papers.head(10)

In [ ]:
# Lorenz curve points (citation concentration)
x, y = metrics.lorenz_curve(df.citation_count.values)
import plotly.graph_objects as go
fig = go.Figure()
fig.add_scatter(x=[0,1], y=[0,1], mode='lines', name='Equality', line=dict(dash='dash'))
fig.add_scatter(x=x, y=y, mode='lines', name='Lorenz')
fig.update_layout(title='Citation Lorenz curve', xaxis_title='Cumulative share of papers', yaxis_title='Cumulative share of citations', height=400)
fig.show()

## 5. Research Pressure Index (Pressure tab)
Four per-year components (volume growth, impact dilution, geographic concentration, topic crowding), min-max normalised across the period, then averaged into a 0–100 score. **Descriptive composite, not a validated benchmark.**

In [ ]:
years = sorted(df.year.unique()); median_cit = df.citation_count.median()
n_buckets = max(2, df.topic_bucket.nunique()); growth = metrics.yoy_growth(df.groupby('year').size().sort_index()).reindex(years)
rows = {}
for yv in years:
    sub = df[df.year == yv]; exy = d.explode_countries(sub)
    rows[yv] = [
        growth.get(yv, np.nan),
        (sub.citation_count < median_cit).mean(),
        metrics.top_n_share(exy.groupby('iso2').size().values, 5)/100 if not exy.empty else np.nan,
        1 - metrics.shannon_entropy(sub.topic_bucket.value_counts().values)/np.log2(n_buckets),
    ]
raw = pd.DataFrame.from_dict(rows, orient='index', columns=['Volume growth','Impact dilution','Geographic concentration','Topic crowding'])
norm = (raw - raw.min()) / (raw.max() - raw.min())
norm['Index'] = (norm.mean(axis=1) * 100).round(0)
norm.round(2)